# Visual Product Search — Exploration Notebook

Use this notebook for:
- Dataset sanity checks
- Visualising query-gallery pairs
- Quick embedding experiments
- Plotting ablation results

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from PIL import Image

from src.utils.helpers import load_config
from src.utils.dataset import load_partition
from src.utils.metrics import evaluate_retrieval, format_metrics

cfg = load_config('../configs/config.yaml')
print('Config loaded.')

## 1. Dataset Statistics

In [ ]:
partition = load_partition(cfg.paths.partition_file)

for split, samples in partition.items():
    item_ids = set(s[1] for s in samples)
    print(f'{split:<8}  images={len(samples):>6}  unique_items={len(item_ids):>5}')

## 2. Visualise a few gallery items

In [ ]:
import random
from collections import defaultdict

img_dir = Path(cfg.paths.img_dir)
gallery = partition['gallery']

# Pick 4 random items and show up to 4 images per item
item_to_imgs = defaultdict(list)
for img_path, item_id in gallery:
    item_to_imgs[item_id].append(img_path)

selected_items = random.sample(list(item_to_imgs.keys()), 4)

fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for row, item_id in enumerate(selected_items):
    imgs = item_to_imgs[item_id][:4]
    for col in range(4):
        ax = axes[row, col]
        if col < len(imgs):
            img = Image.open(img_dir / imgs[col]).convert('RGB')
            ax.imshow(img)
            ax.set_title(f'{item_id[:10]}', fontsize=7)
        ax.axis('off')

plt.suptitle('Gallery: 4 items × up to 4 views', fontsize=13)
plt.tight_layout()
plt.show()

## 3. YOLO Crop Demo

In [ ]:
from src.models.yolo_model import YOLODetector

yolo = YOLODetector(model_name=cfg.yolo.model_name)

sample_path = img_dir / gallery[0][0]
img = Image.open(sample_path).convert('RGB')
crop, bbox = yolo.detect_and_crop(img)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))
ax1.imshow(img);  ax1.set_title('Original');  ax1.axis('off')
ax2.imshow(crop); ax2.set_title(f'YOLO crop  bbox={bbox}'); ax2.axis('off')
plt.tight_layout()
plt.show()

## 4. Embedding Space Visualisation (t-SNE)

In [ ]:
# Load a small subset of gallery embeddings from a saved index
import json
import numpy as np

meta_files = list(Path('../index').glob('metadata_*.json'))
if not meta_files:
    print('No index found. Run build_index.py first.')
else:
    from src.retrieval.indexer import HNSWIndexer
    meta_path  = str(meta_files[0])
    index_path = meta_path.replace('metadata_', 'hnsw_').replace('.json', '.bin')

    indexer = HNSWIndexer(dim=cfg.embedding.embedding_dim)
    indexer.load(index_path, meta_path)

    # Sample 500 embeddings for t-SNE
    N = min(500, indexer.n_items)
    sample_ids = np.random.choice(indexer.n_items, N, replace=False)

    # Retrieve embeddings by re-querying each point against itself
    # (hnswlib doesn't expose raw vectors directly; we use a proxy)
    print(f'Index contains {indexer.n_items} items. Sampling {N} for t-SNE.')
    print('(Implement vector extraction if needed for your hnswlib version.)')

## 5. Ablation Results Comparison

In [ ]:
import json

summary_path = Path('../results/ablation_summary.json')
if not summary_path.exists():
    print('Run run_ablation.py first to generate summary.')
else:
    with open(summary_path) as f:
        summary = json.load(f)

    K_vals   = [5, 10, 15]
    metrics  = ['recall', 'ndcg', 'map']
    cond_keys = list(summary.keys())

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, metric in zip(axes, metrics):
        for cond in cond_keys:
            means = [summary[cond].get(f'{metric}@{k}', {}).get('mean', 0) for k in K_vals]
            stds  = [summary[cond].get(f'{metric}@{k}', {}).get('std', 0)  for k in K_vals]
            ax.errorbar(K_vals, means, yerr=stds, marker='o', label=cond, capsize=4)
        ax.set_title(metric.upper())
        ax.set_xlabel('K')
        ax.set_ylabel(metric)
        ax.set_xticks(K_vals)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    plt.suptitle('Ablation Study: Condition A vs B vs C', fontsize=13)
    plt.tight_layout()
    plt.savefig('../results/ablation_plot.png', dpi=150)
    plt.show()
    print('Plot saved → results/ablation_plot.png')